
# Phase 3 – ML Modeling: French MTPL Insurance Claims


--- 
## Projektübersicht

Dieses Notebook dokumentiert das Python-Modul `phase3_ml_modeling.py` und erklärt die vollständige Machine-Learning-Pipeline für die Modellierung von Versicherungsansprüchen (Motor Third Party Liability – MTPL).

Das Projekt umfasst:

- **Claim Frequency Modeling** → Vorhersage der Anzahl von Schadensfällen
- **Claim Severity Modeling** → Vorhersage der Schadenshöhe
- **Feature Engineering**
- **Monotonic Constraints**
- **Cross Validation**
- **Bayesian Hyperparameter Optimization mit Optuna**
- **Training von LightGBM- und XGBoost-Modellen**
- **Serialisierung der Modelle**
- **Automatisierte Unit Tests**

---

## Verwendete Technologien

| Technologie | Zweck |
|---|---|
| Pandas | Datenverarbeitung |
| NumPy | Numerische Operationen |
| LightGBM | Gradient Boosting |
| XGBoost | Gradient Boosting |
| Optuna | Hyperparameter-Tuning |
| Scikit-Learn | Cross Validation & Preprocessing |
| Pickle | Modell-Speicherung |



# 2. Imports & Konfiguration

Die Datei verwendet moderne ML-Bibliotheken und setzt zusätzlich auf:

- Logging
- Reproduzierbarkeit durch Seeds
- Early Stopping
- saubere Verzeichnisstruktur


In [ ]:

SEED = 42
N_SPLITS = 5
N_TRIALS = 30
EARLY_STOP = 50



## Bedeutung der Konfiguration

| Parameter | Beschreibung |
|---|---|
| `SEED` | Reproduzierbarkeit |
| `N_SPLITS` | Anzahl der CV-Folds |
| `N_TRIALS` | Anzahl der Optuna-Optimierungen |
| `EARLY_STOP` | Verhindert Overfitting |



# 3. Datenvalidierung

Die Funktion `load_data()` übernimmt:

- Laden der CSV-Datei
- Validierung der benötigten Spalten
- Entfernen ungültiger Zeilen
- Konsistenzprüfungen

Dadurch wird sichergestellt, dass das Modell nur mit validen Daten trainiert wird.


In [ ]:

required = {
    "Exposure", "VehPower", "VehAge", "DrivAge",
    "BonusMalus", "VehBrand", "VehGas", "Area",
    "Density", "Region", "ClaimTotal", "ClaimNb",
}



## Validierungslogik

Die Pipeline überprüft:

- Keine negativen Claim Counts
- Keine negativen Claim Costs
- Exposure > 0
- Vollständiges Schema

Dies ist besonders wichtig für:

- Versicherungsmodelle
- regulatorische Compliance
- stabile Trainingsprozesse



# 4. Feature Engineering

Das Feature Engineering erweitert die Rohdaten um zusätzliche informative Variablen.

## Ziel

Die Modelle sollen:

- nicht-lineare Zusammenhänge erkennen
- stark schiefe Verteilungen stabilisieren
- regulatorische Reports ermöglichen


In [ ]:

df["log_Density"] = np.log1p(df["Density"])
df["BonusMalus_sq"] = df["BonusMalus"] ** 2
df["inv_DrivAge"] = 1.0 / df["DrivAge"].clip(lower=18)



## Erzeugte Features

| Feature | Zweck |
|---|---|
| `log_Density` | Stabilisierung schiefer Verteilungen |
| `BonusMalus_sq` | Nicht-lineare Risikoeffekte |
| `inv_DrivAge` | Höheres Risiko junger Fahrer |
| `VehAge_band` | Gruppierung für Reporting |
| `DrivAge_band` | Alterssegmentierung |
| `BonusMalus_band` | Tarifklassen |



# 5. Kategorische Variablen

Baumbasierte Modelle benötigen numerische Eingaben.

Daher werden kategorische Variablen per `LabelEncoder` kodiert.


In [ ]:

CATEGORICALS = ["VehBrand", "VehGas", "Area", "Region"]



## Wichtiges Designziel

Die Encoders werden gespeichert, um:

- Konsistenz zwischen Training und Inference sicherzustellen
- Data Leakage zu verhindern
- unbekannte Kategorien sauber zu behandeln



# 6. Monotonic Constraints

Ein zentrales professionelles Feature dieses Projekts sind monotone Constraints.

Diese werden verwendet, um regulatorische Anforderungen einzuhalten.

## Beispiel

Ein höherer `BonusMalus` darf die Risikoprognose nicht reduzieren.


In [ ]:

MONOTONE_CONSTRAINTS_FREQ = {
    "BonusMalus": 1,
    "DrivAge": 0,
    "VehAge": 0,
    "VehPower": 1,
}



## Vorteile monotonic constraints

- bessere Interpretierbarkeit
- regulatorische Compliance
- stabilere Modelle
- realistischere Vorhersagen



# 7. Cross Validation

Die Pipeline verwendet:

## Stratified 5-Fold Cross Validation

Da Schadensfälle selten sind, wird auf Basis von `ClaimNb` stratifiziert.

Dadurch bleibt die Klassenverteilung in allen Folds stabil.


In [ ]:

skf = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=seed
)



## Warum ist Stratification wichtig

Versicherungsdaten sind typischerweise:

- hochgradig unausgeglichen
- stark skewed
- seltene Ereignisse dominieren

Ohne Stratification wären die Validierungsergebnisse instabil.



# 8. Hyperparameter-Optimierung mit Optuna

Das Projekt nutzt Bayesian Optimization über den TPE-Sampler.

## Vorteile gegenüber Grid Search

- deutlich effizienter
- weniger Trainingsläufe
- intelligentere Suche
- bessere Skalierbarkeit


In [ ]:

sampler = optuna.samplers.TPESampler(seed=SEED)

study = optuna.create_study(
    direction="minimize",
    sampler=sampler
)



## Optimierte Parameter

### LightGBM

- learning_rate
- num_leaves
- max_depth
- subsample
- regularization

### XGBoost

- max_depth
- min_child_weight
- colsample_bytree
- reg_alpha
- reg_lambda



# 9. Modelltraining

Das Projekt trainiert insgesamt vier Modelle:

| Modell | Ziel |
|---|---|
| LightGBM Poisson | Claim Frequency |
| LightGBM Gamma | Claim Severity |
| XGBoost Poisson | Claim Frequency |
| XGBoost Gamma | Claim Severity |



## Warum Poisson & Gamma?

### Poisson
Geeignet für:

- Count Data
- seltene Ereignisse
- Claim Frequencies

### Gamma
Geeignet für:

- positive kontinuierliche Werte
- stark schiefe Kostenverteilungen
- Versicherungs-Schäden



# 10. Modell-Speicherung

Alle Modelle werden serialisiert und als `.pkl` gespeichert.


In [ ]:

with open(path, "wb") as fh:
    pickle.dump(obj, fh, protocol=pickle.HIGHEST_PROTOCOL)



## Gespeicherte Artefakte

- trainierte Modelle
- Feature Engineer
- Hyperparameter
- Pipeline-Metadaten

Dadurch wird das Deployment erheblich vereinfacht.



# 11. Unit Tests

Die Datei enthält integrierte Unit Tests.

Diese prüfen:

- Feature Engineering
- Constraint-Vektoren
- Cross Validation
- Datenvalidierung
- Pickle-Roundtrips



## Beispielhafte Teststrategie

Das Projekt verwendet bewusst:

- kleine Dummy-Datasets
- isolierte Komponenten-Tests
- schnelle Ausführung

Dadurch bleiben die Tests leichtgewichtig und CI/CD-fähig.



# 12. Fazit

Das Projekt implementiert eine vollständige produktionsnahe ML-Pipeline für Versicherungsdaten.

## Technische Stärken

- moderne Gradient-Boosting-Modelle
- robuste Validierung
- regulatorische Constraints
- Hyperparameter-Tuning
- automatisierte Tests
- Deployment-fähige Serialisierung

Die Architektur eignet sich sowohl für:

- Forschung
- Portfolio-Projekte
- produktive ML-Systeme
- actuarial pricing pipelines


------------------------------------------------------------------------

# Kontakt

**Autor:** <marksquant@gmail.com>  
**Projekt:** Tarifmodellierung: GLM vs. Machine Learning; ML Modeling: French MTPL Insurance Claims   
**Sprache:** Python  
**Jahr:** 2026